# Preprocessing after Feature Engineering

This notebook is part of the consolidated workflow. Do not change code logic; headings were added for structure.

## 1. Imports

Common imports used in the preprocessing pipeline.


In [10]:
# 05a_preprocessing_enhanced.ipynb

import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## 2. Load Data

Loading datasets used for preprocessing.


In [11]:
# Load the enhanced dataset
df = pd.read_csv("../data/processed/cardekho_enhanced.csv")

print(f"Enhanced dataset shape: {df.shape}")
df.head()

Enhanced dataset shape: (15411, 22)


,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,...,selling_price,age_squared,km_per_year,power_per_cc,mileage_per_hp,age_km_interaction,engine_power_interaction,engine_category,brand_group,brand_model
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,...,120000,81,12000.000000,0.058093,0.416490,1080000,36854.80,Small,Mass Market,Maruti_Alto
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,...,550000,25,3333.333333,0.068447,0.227711,100000,98154.00,Small,Mass Market,Hyundai_Grand
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,...,215000,121,5000.000000,0.066778,0.209877,660000,95760.00,Small,Mass Market,Hyundai_i20
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,...,226000,81,3700.000000,0.067167,0.307195,333000,66965.80,Small,Mass Market,Maruti_Alto
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,...,570000,36,4285.714286,0.065771,0.228637,180000,147687.82,Small,Mass Market,Ford_Ecosport


## 3. Separate features and target

Splitting data into X and y and train/test sets.


In [12]:
# Define target
target = "selling_price"

# Separate features and target
X = df.drop(columns=[target, 'car_name'])
y = df[target]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")


Train shape: (12328, 20)
Test shape: (3083, 20)


In [13]:
# Define updated categorical features
categorical_features = [
    'brand',            # Existing
    'model',            # Existing
    'seller_type',      # Existing
    'fuel_type',        # Existing
    'transmission_type', # Existing
    'engine_category',  # NEW
    'brand_group',      # NEW
    'brand_model'       # NEW
]

# Define updated numerical features
numerical_features = [
    'vehicle_age',      # Existing
    'km_driven',        # Existing
    'mileage',          # Existing
    'engine',           # Existing
    'max_power',        # Existing
    'seats',            # Existing
    'age_squared',      # NEW
    'km_per_year',      # NEW
    'power_per_cc',     # NEW
    'mileage_per_hp',   # NEW
    'age_km_interaction', # NEW
    'engine_power_interaction' # NEW
]

print(f"\nCategorical features: {len(categorical_features)}")
print(f"Numerical features: {len(numerical_features)}")


Categorical features: 8
Numerical features: 12


## 4. Preprocessing Pipeline

Feature transformations, encoders and scalers.


In [14]:
# Create transformers
categorical_transformer = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

numerical_transformer = StandardScaler()

# Combine into preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Fit and transform
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"\nProcessed train shape: {X_train_processed.shape}")
print(f"Processed test shape: {X_test_processed.shape}")



Processed train shape: (12328, 295)
Processed test shape: (3083, 295)


In [15]:
# Get feature names
feature_names = preprocessor.get_feature_names_out()
print(f"\nTotal features: {len(feature_names)}")

# Convert to DataFrames
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)


Total features: 295


In [16]:
# Save processed datasets
train_processed = X_train_processed_df.copy()
train_processed['selling_price'] = y_train

test_processed = X_test_processed_df.copy()
test_processed['selling_price'] = y_test

train_processed.to_csv('../data/processed/train_enhanced_processed.csv', index=False)
test_processed.to_csv('../data/processed/test_enhanced_processed.csv', index=False)

# Save preprocessor
joblib.dump(preprocessor, '../models/preprocessor_enhanced.pkl')

print("\n✅ Preprocessing complete!")
print(f"Training data saved: {train_processed.shape}")
print(f"Testing data saved: {test_processed.shape}")
print(f"Preprocessor saved: ../models/preprocessor_enhanced.pkl")


✅ Preprocessing complete!
Training data saved: (12328, 296)
Testing data saved: (3083, 296)
Preprocessor saved: ../models/preprocessor_enhanced.pkl
